# OCR Lab — Google Colab (L4 / A100 + Google Drive)

**Akış:** Telefon (CamScanner PDF) → **Google Drive `ocr-lab/input/`** → Colab GPU → **Google Drive `ocr-lab/output/`**

- **Önerilen GPU:** **L4 (24 GB)** (fiyat/performans açısından en verimli; `dots.mocr` tam performans çalışır). Büyük toplu kitaplar için **A100** de seçilebilir.
- **T4 notu:** T4 GPU'su `fp8` desteklemez; `dots.mocr` ve `deepseek-ocr2` için **L4** veya **A100** seçin (T4'te `glm-ocr` veya `lightonocr` çalışır).
- **Kod güncelleme:** Tüm mantık repodaki `colab/runner.py` ve `src/ocrnow/` içindedir. Yerelden `git push` yaptığınızda Hücre 2 otomatik `git pull` çeker.
- **Telefondan tek dokunuş:** Ayar gerektirmeyen tek hücreli `colab/telefon.ipynb` defterini kullanın (bkz. README).


In [ ]:
#@title 1. Google Drive'ı Bağla ve Repoyu Eşitle
import importlib, subprocess, sys
from pathlib import Path
from google.colab import drive

drive_klasoru = "/content/drive/MyDrive/ocr-lab"  #@param {type:"string"}
repo_url = "https://github.com/NECATKO/ocr-lab.git"  #@param {type:"string"}
branch = "main"  #@param {type:"string"}

drive.mount("/content/drive")

LOCAL_ROOT = Path("/content/ocr-lab")
if not (LOCAL_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "-b", branch, repo_url, str(LOCAL_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(LOCAL_ROOT), "pull", "--ff-only"], check=True)

if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))
from colab import runner
importlib.reload(runner)

runner.prepare_drive(drive_klasoru)

In [ ]:
#@title 2. Ortam ve Model Kurulumu (oturum başına 1 kez)
kurulacak_model = "dots.mocr"  #@param ["dots.mocr", "glm-ocr", "lightonocr", "deepseek-ocr2", "onerilenler", "tesseract"]

runner.setup_env(drive_klasoru, model=kurulacak_model)

In [ ]:
#@title 3. OCR Çalıştır ve Sonuçları Drive'a Kaydet
dosya = "HEPSI"  #@param {type:"string"}
sayfalar = "hepsi"  #@param {type:"string"}
profil = "kitap"  #@param ["kitap", "ders-notu", "gunluk", "fatura", "yazma"]
model = "dots.mocr"  #@param ["dots.mocr", "glm-ocr", "lightonocr", "deepseek-ocr2", "tesseract"]
okuma = "duzen"  #@param ["duzen", "duz"]
formatlar = "docx txt"  #@param {type:"string"}
ilk_sayfa = 0  #@param {type:"integer"}
isaretle = True  #@param {type:"boolean"}
kesme = True  #@param {type:"boolean"}
dpi = 200  #@param {type:"integer"}
tam_kalite_bf16 = True  #@param {type:"boolean"}
islenenleri_tasi = False  #@param {type:"boolean"}

runner.run_ocr(
    drive_klasoru,
    dosya=dosya,
    sayfalar=sayfalar,
    profil=profil,
    model=model,
    okuma=okuma,
    formatlar=formatlar,
    ilk_sayfa=ilk_sayfa,
    isaretle=isaretle,
    kesme=kesme,
    dpi=dpi,
    tam_kalite_bf16=tam_kalite_bf16,
    islenenleri_tasi=islenenleri_tasi,
)

### İpucu: GPU harcamadan `kesme.txt` güncelleme veya yeni format alma

Bir kitabı bir kez okuttuktan sonra sayfa OCR çıktıları `MyDrive/ocr-lab/cache/` içinde saklanır.
`MyDrive/ocr-lab/kesme.txt` dosyasına yeni kelimeler ekleyip `.docx` / `.txt` dosyasını yeniden almak ya da farklı format (`md`, `pdf`) çıkarmak için **CPU çalışma zamanında** bile Hücre 1 → Hücre 2 (`kurulacak_model = "tesseract"`) → Hücre 3'ü çalıştırabilirsiniz; önbellekteki sayfalar için GPU kullanılmaz.